# Lab 3 · Exercise C — Two channels with correlated background
We combine two search channels ($e$ and $\mu$), each with expected background $\beta_0 = 100 \pm 15$ ($15\%$) and expected signal $s = 25$ (for $\mu = 1$).
We observe $n_e = 130$ and $n_\mu = 128$ events.

We encode the background uncertainty in a $2 \times 2$ covariance matrix $\Sigma_b(\rho)$, factorise it as $\Sigma_b = A A^T$, and study how the correlation $\rho \in \{0, 0.5, 1\}$ affects the measurement of $\mu$ and the discovery significance $Z_0$.


In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name


## C1 · Covariance matrix and diagonalisation
For two channels with background uncertainties $\sigma_e = \sigma_\mu = 15$ and correlation $\rho$:
$$\Sigma_b(\rho) = \begin{pmatrix} \sigma_e^2 & \rho\,\sigma_e\sigma_\mu \\ \rho\,\sigma_e\sigma_\mu & \sigma_\mu^2 \end{pmatrix}$$
Diagonalise $\Sigma_b$ with `np.linalg.eigh` and build $A = U\sqrt{\Lambda}$ keeping **only columns with non-zero eigenvalues** ($\lambda_k > 10^{-10}$), so that $A A^T = \Sigma_b$.


In [ ]:
b0_e, b0_mu   = 100.0, 100.0
sb_e, sb_mu   = 15.0,  15.0
s0_e, s0_mu   = 25.0,  25.0
n_e_obs, n_mu_obs = 130, 128

# Example: rho = 0.0 (uncorrelated)
rho = 0.0
Sigma_0 = np.array([
    [sb_e**2,           rho * sb_e * sb_mu],
    [rho * sb_e * sb_mu, sb_mu**2         ]
])
evals_0, U_0 = np.linalg.eigh(Sigma_0)
print("rho = 0.0 eigenvalues:", evals_0)

# TODO: select eigenvalues > 1e-10 and compute A_0 = U_0[:, keep] * np.sqrt(evals_0[keep])
A_0 = None   # TODO

# Also check rho = 1.0 (fully correlated -> rank 1!)
Sigma_1 = np.array([
    [sb_e**2,  sb_e * sb_mu],
    [sb_e * sb_mu, sb_mu**2]
])
evals_1, U_1 = np.linalg.eigh(Sigma_1)
print("rho = 1.0 eigenvalues:", evals_1)
A_1 = U_1[:, evals_1 > 1e-10] * np.sqrt(evals_1[evals_1 > 1e-10])

In [ ]:
check("A_0 implemented", A_0 is not None)
check("A_0 A_0^T reproduces Sigma_0", np.allclose(A_0 @ A_0.T, Sigma_0, atol=1e-10))
check("rho = 0 needs 2 nuisance parameters (rank 2)", A_0.shape[1] == 2)
check("rho = 1 needs 1 nuisance parameter (rank 1)",  A_1.shape[1] == 1)

## C2 · Build the RooFit model for $\rho = 0$
For $\rho = 0$, $A_0$ has shape $(2, 2)$, so we introduce two independent unit-normal nuisance parameters $\theta_0, \theta_1$:
$$\beta_e = b_{0e} + A_{00}\theta_0 + A_{01}\theta_1, \qquad \beta_\mu = b_{0\mu} + A_{10}\theta_0 + A_{11}\theta_1$$
Each $\theta_k$ is constrained by $\text{Gauss}(g_k \mid \theta_k, 1)$ with global observable $g_k = 0$.
Complete `beta_e`, `beta_mu`, `lam_e`, and `lam_mu` below, then run RooStats to find $\hat{\mu}$, the 68% interval, and $Z_0$.

In [ ]:
# Parameter of interest, nuisance parameters, and global observables
mu     = ROOT.RooRealVar("mu", "signal strength", 1.0, 0.0, 5.0)
theta0 = ROOT.RooRealVar("theta0", "nuisance 0", 0.0, -5.0, 5.0)
theta1 = ROOT.RooRealVar("theta1", "nuisance 1", 0.0, -5.0, 5.0)
g0     = ROOT.RooRealVar("g0", "global obs 0", 0.0, -5.0, 5.0); g0.setConstant(True)
g1     = ROOT.RooRealVar("g1", "global obs 1", 0.0, -5.0, 5.0); g1.setConstant(True)
one    = ROOT.RooRealVar("one", "unit width", 1.0, 0.1, 10.0); one.setConstant(True)

# 1D Gaussian constraints
c0 = ROOT.RooGaussian("c0", "c0", g0, theta0, one)
c1 = ROOT.RooGaussian("c1", "c1", g1, theta1, one)

# TODO: linear background yields and total expectations lam = mu * s0 + beta
expr_be  = f"{b0_e}  + ({A_0[0,0]})*@0 + ({A_0[0,1]})*@1"
expr_bmu = f"{b0_mu} + ({A_0[1,0]})*@0 + ({A_0[1,1]})*@1"

beta_e  = None   # TODO: ROOT.RooFormulaVar("beta_e",  expr_be,  ROOT.RooArgList(theta0, theta1))
beta_mu = None   # TODO: ROOT.RooFormulaVar("beta_mu", expr_bmu, ROOT.RooArgList(theta0, theta1))
lam_e   = None   # TODO: ROOT.RooFormulaVar("lam_e",  f"max(@0*{s0_e}  + @1, 1e-9)", ROOT.RooArgList(mu, beta_e))
lam_mu  = None   # TODO: ROOT.RooFormulaVar("lam_mu", f"max(@0*{s0_mu} + @1, 1e-9)", ROOT.RooArgList(mu, beta_mu))

# Observables and Poisson terms
n_e    = ROOT.RooRealVar("n_e",  "n_e",  0, 500)
n_mu   = ROOT.RooRealVar("n_mu", "n_mu", 0, 500)
pois_e  = ROOT.RooPoisson("pois_e",  "pois_e",  n_e,  lam_e)
pois_mu = ROOT.RooPoisson("pois_mu", "pois_mu", n_mu, lam_mu)

# Full PDF and dataset
pdf  = ROOT.RooProdPdf("model", "model", ROOT.RooArgList(pois_e, pois_mu, c0, c1))
obs  = ROOT.RooArgSet(n_e, n_mu)
data = ROOT.RooDataSet("data", "data", obs)
n_e.setVal(n_e_obs); n_mu.setVal(n_mu_obs)
data.add(obs)

In [ ]:
# Workspace and ModelConfig
w = ROOT.RooWorkspace("w_rho0")
w.Import(pdf)
w.Import(data)

mc = ROOT.RooStats.ModelConfig("mc", w)
mc.SetPdf(w.pdf("model"))
mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("mu")))
mc.SetNuisanceParameters(ROOT.RooArgSet(w.var("theta0"), w.var("theta1")))
mc.SetGlobalObservables(ROOT.RooArgSet(w.var("g0"), w.var("g1")))
mc.SetObservables(ROOT.RooArgSet(w.var("n_e"), w.var("n_mu")))

# Fit, 68% interval, and significance Z0
w_data = w.data("data")
w_mu   = w.var("mu")
w.pdf("model").fitTo(w_data, ROOT.RooFit.GlobalObservables(mc.GetGlobalObservables()), ROOT.RooFit.PrintLevel(-1))
mu_hat = w_mu.getVal()

plc = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
plc.SetConfidenceLevel(0.683)
iv = plc.GetInterval()
lo, hi = iv.LowerLimit(w_mu), iv.UpperLimit(w_mu)

w_mu.setVal(1.0)
plc_null = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
null_poi = mc.GetParametersOfInterest().snapshot()
null_poi.first().setVal(0.0)
plc_null.SetNullParameters(null_poi)
Z0 = plc_null.GetHypoTest().Significance()


In [ ]:
print(f"rho = 0.0: mu_hat = {mu_hat:.2f}  [{lo:.2f}, {hi:.2f}]   Z0 = {Z0:.2f}")
check("mu_hat close to 1.16", abs(mu_hat - 1.16) < 0.15)
check("significance Z0 around 2.2", abs(Z0 - 2.20) < 0.25)

## C3 · Scan the background correlation ($\rho = 0,\, 0.5,\, 1$)
Now compare uncorrelated ($\rho = 0$), partially correlated ($\rho = 0.5$), and fully correlated ($\rho = 1$) background uncertainties.
Notice how for $\rho = 1$, `A` automatically drops from 2 columns to **1 column** (a single shared nuisance parameter $\theta_0$).

In [ ]:
results = {}
for idx, rho in enumerate((0.0, 0.5, 1.0)):
    Sigma = np.array([
        [sb_e**2,           rho * sb_e * sb_mu],
        [rho * sb_e * sb_mu, sb_mu**2         ]
    ])
    evals, U = np.linalg.eigh(Sigma)
    keep = evals > 1e-10
    A = U[:, keep] * np.sqrt(evals[keep])
    n_nuis = A.shape[1]

    mu     = ROOT.RooRealVar("mu", "signal strength", 1.0, 0.0, 5.0)
    thetas = [ROOT.RooRealVar(f"theta{k}", f"theta{k}", 0.0, -5.0, 5.0) for k in range(n_nuis)]
    globs  = [ROOT.RooRealVar(f"g{k}",     f"g{k}",     0.0, -5.0, 5.0) for k in range(n_nuis)]
    for g in globs:
        g.setConstant(True)

    one    = ROOT.RooRealVar("one", "unit width", 1.0, 0.1, 10.0); one.setConstant(True)
    constr = [ROOT.RooGaussian(f"c{k}", f"c{k}", g, th, one) for k, (th, g) in enumerate(zip(thetas, globs))]

    th_list  = ROOT.RooArgList(*thetas)
    expr_be  = f"{b0_e}"  + "".join(f" + ({A[0,k]})*@{k}" for k in range(n_nuis))
    expr_bmu = f"{b0_mu}" + "".join(f" + ({A[1,k]})*@{k}" for k in range(n_nuis))

    beta_e  = ROOT.RooFormulaVar("beta_e",  expr_be,  th_list)
    beta_mu = ROOT.RooFormulaVar("beta_mu", expr_bmu, th_list)
    lam_e   = ROOT.RooFormulaVar("lam_e",  f"max(@0*{s0_e}  + @1, 1e-9)", ROOT.RooArgList(mu, beta_e))
    lam_mu  = ROOT.RooFormulaVar("lam_mu", f"max(@0*{s0_mu} + @1, 1e-9)", ROOT.RooArgList(mu, beta_mu))

    n_e     = ROOT.RooRealVar("n_e",  "n_e",  0, 500)
    n_mu    = ROOT.RooRealVar("n_mu", "n_mu", 0, 500)
    pois_e  = ROOT.RooPoisson("pois_e",  "pois_e",  n_e,  lam_e)
    pois_mu = ROOT.RooPoisson("pois_mu", "pois_mu", n_mu, lam_mu)

    pdf  = ROOT.RooProdPdf("model", "model", ROOT.RooArgList(pois_e, pois_mu, *constr))
    obs  = ROOT.RooArgSet(n_e, n_mu)
    data = ROOT.RooDataSet("data", "data", obs)
    n_e.setVal(n_e_obs); n_mu.setVal(n_mu_obs)
    data.add(obs)

    w = ROOT.RooWorkspace(f"w_scan_{idx}")
    w.Import(pdf)
    w.Import(data)

    mc = ROOT.RooStats.ModelConfig("mc", w)
    mc.SetPdf(w.pdf("model"))
    mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("mu")))
    mc.SetNuisanceParameters(ROOT.RooArgSet(*(w.var(th.GetName()) for th in thetas)))
    mc.SetGlobalObservables(ROOT.RooArgSet(*(w.var(g.GetName()) for g in globs)))
    mc.SetObservables(ROOT.RooArgSet(w.var("n_e"), w.var("n_mu")))

    w_data = w.data("data")
    w_mu   = w.var("mu")
    w.pdf("model").fitTo(w_data, ROOT.RooFit.GlobalObservables(mc.GetGlobalObservables()), ROOT.RooFit.PrintLevel(-1))
    mu_hat = w_mu.getVal()

    plc = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
    plc.SetConfidenceLevel(0.683)
    iv = plc.GetInterval()
    lo, hi = iv.LowerLimit(w_mu), iv.UpperLimit(w_mu)

    w_mu.setVal(1.0)
    plc_null = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
    null_poi = mc.GetParametersOfInterest().snapshot()
    null_poi.first().setVal(0.0)
    plc_null.SetNullParameters(null_poi)
    Z0 = plc_null.GetHypoTest().Significance()

    results[rho] = (mu_hat, lo, hi, Z0)
    print(f"rho = {rho:.1f} (rank = {n_nuis}): mu_hat = {mu_hat:.2f}  [{lo:.2f}, {hi:.2f}]  width = {hi-lo:.2f}   Z0 = {Z0:.2f}")



In [ ]:
widths = {rho: r[2] - r[1] for rho, r in results.items()}
check("interval width increases with background correlation", widths[1.0] > widths[0.0])
check("significance Z0 drops when background errors are correlated", results[1.0][3] < results[0.0][3] - 0.3)


## Questions (3–4 lines each)
1. How many nuisance parameters ($\theta_k$) does the model use for $\rho = 0$ versus $\rho = 1$? Why?
2. Compare the 68% interval width on $\mu$ between $\rho = 0$ and $\rho = 1$: why does positive correlation make the measurement less precise?
3. Why does the significance $Z_0$ drop significantly when $\rho$ goes from $0$ to $1$? (Hint: what happens when both backgrounds fluctuate upward together under $\mu = 0$?)
4. What would happen to $Z_0$ if the correlation $\rho$ were instead on the **signal** uncertainty rather than on the background?

